# 19-Site Depth Optimization Plots

This notebook plots energy, fidelity, and bond-correlation delocalization from continuous edge-colored HVA optimization runs.

For serious runs, execute from the project root:

```powershell
python scripts/legacy/run_19site_edge_colored_depth_optimization.py --init rvb --coverings 24 --max-layers 4 --maxfev 500 --restarts 6
python scripts/legacy/run_19site_edge_colored_depth_optimization.py --init static --max-layers 4 --maxfev 500 --restarts 6
```

The notebook does not launch those long jobs by default.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

EXACT_ENERGY = float(np.load(PROJECT / "results" / "19site_fixed_sz_exact_n9.npz")["energy"])
plt.rcParams["figure.dpi"] = 130

## Optional Smoke Run

In [ ]:
RUN_SMOKE = False

if RUN_SMOKE:
    for init in ["static", "rvb"]:
        cmd = [
            sys.executable,
            str(PROJECT / "scripts" / "legacy" / "run_19site_edge_colored_depth_optimization.py"),
            "--init",
            init,
            "--max-layers",
            "1",
            "--maxfev",
            "4",
            "--restarts",
            "0",
        ]
        if init == "rvb":
            cmd.extend(["--coverings", "4"])
        result = subprocess.run(cmd, cwd=PROJECT, text=True, capture_output=True, check=True)
        print(result.stdout)

## Load Results

In [ ]:
paths = sorted((PROJECT / "results").glob("19site_edge_colored_depth_optimization_*.csv"))
paths = [path for path in paths if not path.name.endswith("_history.csv")]

if not paths:
    raise FileNotFoundError("Run the depth optimization script first.")

frames = []
for path in paths:
    frame = pd.read_csv(path)
    frame["source_file"] = path.name
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)
df[[
    "initialization",
    "depth",
    "energy",
    "error_vs_reference",
    "fidelity",
    "af_weight_participation",
    "max_magnetization",
    "evaluations",
]]

## Energy, Fidelity, and Delocalization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))

for init, group in df.groupby("initialization"):
    group = group.sort_values("depth")
    axes[0].plot(group["depth"], group["energy"], marker="o", label=init)
    axes[1].plot(group["depth"], group["fidelity"], marker="o", label=init)
    axes[2].plot(group["depth"], group["af_weight_participation"], marker="o", label=init)

axes[0].axhline(EXACT_ENERGY, color="black", ls="--", lw=1, label="exact")
axes[0].set_ylabel("Energy")
axes[1].set_ylabel("Fidelity")
axes[2].set_ylabel("AF bond participation")

for ax in axes:
    ax.set_xlabel("Depth p")
    ax.legend(fontsize=8)

fig.tight_layout()
plt.show()